# 🌾 PaddyVLM — Hands-On Workshop

**Build a domain-adapted Vision-Language Model for paddy disease analysis — entirely in Google Colab.**

### Pipeline overview
1. Download the Paddy Disease dataset from Kaggle (automated)
2. Clone expert knowledge resources from GitHub (automated)
3. **Stage 1** — Generate image descriptions with LLaVA-1.6-Mistral-7B (HF · 4-bit NF4)
4. **Stage 2** — Generate multi-turn Q&A dialogs with Mistral-7B-Instruct (HF · 4-bit)
5. **Stage 3** — Generate simple Q&A pairs
6. Convert to LLaVA instruction-tuning format
7. Fine-tune LLaVA-1.5-7B with LoRA

---

### ⚙️ One-time setup (do this BEFORE running any cell)

**Step 1 — GPU runtime**  
`Runtime → Change runtime type → T4 GPU`

**Step 2 — Colab Secrets** (click the 🔑 key icon in the left sidebar, enable notebook access for each):

| Secret Name | Where to get it |
|---|---|
| `KAGGLE_USERNAME` | [kaggle.com/settings/account](https://www.kaggle.com/settings/account) → API section |
| `KAGGLE_KEY` | Same page → *Create New Token* → opens `kaggle.json`, copy the `key` value |
| `HF_TOKEN` | [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → *New token (read)* |


---

> Run cells **top to bottom**.  After Cell 2 you will be asked to **restart the runtime** — do so, then continue from Cell 3.


## Cell 1 — Check GPU & Runtime

In [ ]:
import torch, platform, sys

if not torch.cuda.is_available():
    raise SystemError(
        "No GPU detected. Go to Runtime → Change runtime type → GPU and re-run."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU    : {gpu_name}")
print(f"   VRAM  : {vram_gb:.1f} GB")
print(f"   CUDA  : {torch.version.cuda}")
print(f"   Python: {sys.version.split()[0]}")
print(f"   OS    : {platform.platform()}")


## Cell 2 — Install All Python Dependencies

Pins mutually-compatible versions for Colab (Python 3.12 / CUDA 12.x).  
**After this cell finishes, restart the runtime** (`Runtime → Restart session`),  
then continue from Cell 3.


In [ ]:
!pip install -q --upgrade pip
!pip install -q --upgrade \
    "transformers==4.46.3" \
    "accelerate==1.2.1" \
    "bitsandbytes" \
    "sentencepiece" \
    "peft==0.14.0" \
    "trl==0.12.1" \
    "datasets" \
    "einops" \
    "Pillow>=9.0,<12.0" \
    "kaggle==1.5.12" \
    "huggingface_hub"

print("\n✅ Packages installed.")
print("   👉  Now go to Runtime → Restart session, then run from Cell 3.")


## Cell 3 — Verify Installed Versions

In [ ]:
import transformers, peft, trl, bitsandbytes, torch
print("transformers :", transformers.__version__)
print("peft         :", peft.__version__)
print("trl          :", trl.__version__)
print("bitsandbytes :", bitsandbytes.__version__)
print("torch        :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## Cell 4 — Clone Workshop Repo (gets `other_resources`)



In [ ]:
import os, subprocess

GITHUB_REPO_URL = "https://github.com/rajashikdatta/paddy-vlm.git"

WORKSHOP_DIR    = "/content/paddy-vlm-workshop"
OTHER_RES_DIR   = f"{WORKSHOP_DIR}/other_resources"

if os.path.exists(WORKSHOP_DIR):
    print(f"Repo already at {WORKSHOP_DIR} — skipping clone.")
else:
    print(f"Cloning {GITHUB_REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", GITHUB_REPO_URL, WORKSHOP_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError("git clone failed. Check your GITHUB_REPO_URL above.")
    print(f"✅ Cloned to {WORKSHOP_DIR}")

print("\nother_resources structure:")
!find "{OTHER_RES_DIR}" -maxdepth 3 | sort | head -40


## Cell 5 — Download Paddy Disease Dataset from Kaggle

Credentials are read from **Colab Secrets** — no file uploads needed.

> ⚠️ **Before running this cell**, you must accept the competition rules on Kaggle:  
> → Go to [kaggle.com/competitions/paddy-disease-classification](https://www.kaggle.com/competitions/paddy-disease-classification), scroll down and click **"I Understand and Accept"**.  
> Without this step the API returns 401 even with valid credentials.


In [ ]:

import os, glob, shutil, json
from google.colab import userdata

!pip install -q "kaggle==1.5.12"

# ── Read & validate credentials from Colab Secrets ───────────────────────────
raw_username = userdata.get("KAGGLE_USERNAME").strip()
raw_key      = userdata.get("KAGGLE_KEY").strip()

# Handle the common mistake: user pasted the full kaggle.json as the KEY secret
# e.g. KAGGLE_KEY = '{"username":"foo","key":"abc123"}'
username, key = raw_username, raw_key
if raw_key.startswith("{"):
    try:
        parsed = json.loads(raw_key)
        username = parsed.get("username", raw_username).strip()
        key      = parsed.get("key", raw_key).strip()
        print("ℹ️  Detected full kaggle.json pasted into KAGGLE_KEY — extracted fields.")
    except json.JSONDecodeError:
        pass
elif raw_username.startswith("{"):
    try:
        parsed = json.loads(raw_username)
        username = parsed.get("username", raw_username).strip()
        key      = parsed.get("key", raw_key).strip()
        print("ℹ️  Detected full kaggle.json pasted into KAGGLE_USERNAME — extracted fields.")
    except json.JSONDecodeError:
        pass

print(f"   username : {username}")
print(f"   key      : {key[:4]}{'*' * (len(key)-4)}  (length={len(key)})")

# Validate key looks plausible (Kaggle API keys are 32-char hex strings)
if len(key) < 10:
    raise ValueError(
        f"KAGGLE_KEY looks wrong (length={len(key)}). "
        "Go to kaggle.com/settings → API → Create New Token → open the downloaded "
        "kaggle.json and copy ONLY the value of the 'key' field into the Colab Secret."
    )

# ── Write ~/.kaggle/kaggle.json ───────────────────────────────────────────────
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_json_path, "w") as kf:
    json.dump({"username": username, "key": key}, kf)
os.chmod(kaggle_json_path, 0o600)
print(f"\n✅ Credentials written to {kaggle_json_path}")

DOWNLOAD_DIR = "/content/kaggle_data"
DATASET_DIR  = "/content/paddy-vlm-workshop/datasets/paddy_disease"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print("\nDownloading Paddy Disease Classification dataset from Kaggle ...")
!kaggle competitions download -c paddy-disease-classification -p {DOWNLOAD_DIR}

# ── Check download succeeded ──────────────────────────────────────────────────
zip_files = glob.glob(f"{DOWNLOAD_DIR}/*.zip")
if not zip_files:
    print("\nDownload directory contents:")
    !ls -lh {DOWNLOAD_DIR}
    raise FileNotFoundError(
        "No zip file downloaded.\n"
        "If you see '401 Unauthorized': your API key is wrong or expired.\n"
        "  → Go to kaggle.com/settings → API → Expire Token, then Create New Token.\n"
        "  → Open the downloaded kaggle.json, copy ONLY the 'key' value into the KAGGLE_KEY secret.\n"
        "If you see '403 Forbidden': competition rules not accepted.\n"
        "  → Go to kaggle.com/competitions/paddy-disease-classification and accept the rules."
    )

print("\nUnzipping ...")
for z in zip_files:
    print(f"  Extracting {os.path.basename(z)} ...")
    !unzip -q "{z}" -d {DOWNLOAD_DIR}

print("\nExtracted contents:")
!ls {DOWNLOAD_DIR}

# ── Find the train_images folder ──────────────────────────────────────────────
candidates = [
    f"{DOWNLOAD_DIR}/train_images",
    f"{DOWNLOAD_DIR}/paddy-disease-classification/train_images",
    f"{DOWNLOAD_DIR}/paddy_disease",
    f"{DOWNLOAD_DIR}/train",
]
src = next((c for c in candidates if os.path.exists(c)), None)

if src is None:
    for d in glob.glob(f"{DOWNLOAD_DIR}/**/", recursive=True):
        subs = [x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))]
        if any(c in subs for c in ["blast", "tungro", "healthy", "hispa", "normal"]):
            src = d.rstrip("/")
            break

if src is None:
    print("Could not auto-locate dataset. Full directory tree:")
    !find {DOWNLOAD_DIR} -maxdepth 4 | sort
    raise FileNotFoundError("Dataset folder not found — check the output above.")

os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.exists(DATASET_DIR):
    print(f"\nDataset already at {DATASET_DIR} — skipping move.")
else:
    shutil.move(src, DATASET_DIR)
    print(f"\n✅ Moved {src} → {DATASET_DIR}")

print("\nClass folders:")
for cls in sorted(os.listdir(DATASET_DIR)):
    cls_path = os.path.join(DATASET_DIR, cls)
    if os.path.isdir(cls_path):
        n = len(os.listdir(cls_path))
        print(f"  {cls}: {n} images")


## Cell 6 — (Optional) Mount Google Drive for Persistence

If you skip this cell, outputs are saved locally in `/content/` and will be lost when the runtime disconnects.  
Mounting Drive lets you resume without re-generating data.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    DRIVE_OUT = "/content/drive/MyDrive/PaddyVLM_outputs"
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print(f"✅ Drive mounted. Outputs folder: {DRIVE_OUT}")
except Exception as e:
    DRIVE_OUT = "/content/PaddyVLM_outputs"
    import os; os.makedirs(DRIVE_OUT, exist_ok=True)
    print(f"Drive not mounted (reason: {e}). Using local folder: {DRIVE_OUT}")


## Cell 7 — HuggingFace Login + Load LLaVA-1.6-Mistral-7B (4-bit)

Replaces the original Ollama `llava:13b` call.  
Uses `llava-hf/llava-v1.6-mistral-7b-hf` with BitsAndBytes NF4 4-bit quantisation — fits on a T4 (16 GB VRAM).


In [ ]:
from google.colab import userdata
from huggingface_hub import login
import torch
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig

# ── HuggingFace login (reads HF_TOKEN from Colab Secrets) ────────────────────
hf_token = userdata.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("✅ Logged in to HuggingFace.")
else:
    print("ℹ️  HF_TOKEN secret not set — will attempt to load public models only.")

# ── Load LLaVA ────────────────────────────────────────────────────────────────
LLAVA_MODEL_ID = "llava-hf/llava-v1.6-mistral-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading {LLAVA_MODEL_ID} in 4-bit NF4 ...")
llava_processor = LlavaNextProcessor.from_pretrained(LLAVA_MODEL_ID)
llava_model = LlavaNextForConditionalGeneration.from_pretrained(
    LLAVA_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
llava_model.eval()
print("✅ LLaVA loaded.")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## Cell 8 — Stage 1: Generate Image Descriptions with LLaVA

`MAX_IMAGES_PER_CLASS = 10` → optimised for free Colab T4 GPU limit (~10–15 min total for 10 classes).  
Set to `50` for a richer dataset, or `None` for the full ~10 k-image run (several hours).

**Resume-safe** — re-running this cell skips images already written to the output file,  
so a session disconnect does **not** lose prior progress.


In [ ]:
import os, json, torch
from pathlib import Path
from PIL import Image

# ── Paths ─────────────────────────────────────────────────────────────────────
WORKSHOP_DIR   = "/content/paddy-vlm-workshop"
DATASET_DIR    = Path(f"{WORKSHOP_DIR}/datasets")
ATTRIBUTES_DIR = Path(f"{WORKSHOP_DIR}/other_resources/attributes")
TEMP_DESC_FILE = Path(f"{WORKSHOP_DIR}/paddy_disease_desc.jsonl")

# ── Config ────────────────────────────────────────────────────────────────────
# 10 images × 10 classes = 100 images — fits free Colab T4 in ~10–15 min.
# Raise to 50 for a richer dataset (needs ~1 h), or None for full run.
MAX_IMAGES_PER_CLASS = 10
MAX_NEW_TOKENS_DESC  = 200   # shorter = faster; 200 chars is plenty for a description

def read_txt(path):
    p = Path(path)
    if not p.exists():
        return ""
    for enc in ("utf-8", "latin-1"):
        try:
            return p.read_text(encoding=enc).strip()
        except UnicodeDecodeError:
            pass
    with open(p, "rb") as f:
        return f.read().decode(errors="ignore").strip()

def generate_description(image_path, class_label, dataset_name, attributes):
    image = Image.open(image_path).convert("RGB")
    conversation = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": (
                f"You are an agricultural assistant. Describe this image of a {class_label} "
                f"from the {dataset_name} dataset. "
                f"Use the following attributes for a more detailed description:\n{attributes}"
            )},
        ],
    }]
    prompt = llava_processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = llava_processor(
        text=prompt, images=image, return_tensors="pt", padding=True
    ).to(llava_model.device)
    with torch.inference_mode():
        out = llava_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS_DESC, do_sample=False)
    generated = out[:, inputs["input_ids"].shape[1]:]
    return llava_processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

# ── Resume: load already-processed image paths ────────────────────────────────
processed = set()
if TEMP_DESC_FILE.exists():
    with open(TEMP_DESC_FILE, "r", encoding="utf-8") as _f:
        for _line in _f:
            try:
                processed.add(json.loads(_line)["image_path"])
            except Exception:
                pass
if processed:
    print(f"ℹ️  Resuming — {len(processed)} images already done, skipping them.")

print(f"Stage 1 — writing descriptions to {TEMP_DESC_FILE}")
total = 0
with open(TEMP_DESC_FILE, "a", encoding="utf-8") as out_f:   # "a" = append for resume
    for dataset_dir in DATASET_DIR.iterdir():
        if not dataset_dir.is_dir():
            continue
        dataset_name = dataset_dir.name
        for class_dir in sorted(dataset_dir.iterdir()):
            if not class_dir.is_dir():
                continue
            class_label = class_dir.name
            attr_text   = read_txt(ATTRIBUTES_DIR / dataset_name / f"{class_label}.txt")
            images = [p for p in class_dir.iterdir()
                      if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
            if MAX_IMAGES_PER_CLASS:
                images = images[:MAX_IMAGES_PER_CLASS]
            for img_path in images:
                rel = str(img_path.relative_to(DATASET_DIR).as_posix())
                if rel in processed:
                    continue   # already done — skip
                try:
                    desc = generate_description(img_path, class_label, dataset_name, attr_text)
                    record = {
                        "image_path" : rel,
                        "dataset"    : dataset_name,
                        "class"      : class_label,
                        "attributes" : attr_text,
                        "description": desc,
                    }
                    out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                    out_f.flush()   # ensure progress is saved even on disconnect
                    total += 1
                    if total % 10 == 0:
                        print(f"  [{total}] {class_label}/{img_path.name}")
                except Exception as e:
                    print(f"  ERROR {img_path.name}: {e}")

print(f"\n✅ Stage 1 complete — {total} new descriptions written  "
      f"({len(processed) + total} total in file).")


## Cell 9 — Unload LLaVA, Load Mistral-7B-Instruct (4-bit)

We free LLaVA's VRAM first, then load `mistralai/Mistral-7B-Instruct-v0.2` in 4-bit.  
Replaces the original Ollama `mistral` call.


In [ ]:
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Unloading LLaVA ...")
del llava_model, llava_processor
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after unload: {torch.cuda.memory_allocated()/1e9:.2f} GB")

MISTRAL_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"\nLoading {MISTRAL_MODEL_ID} in 4-bit NF4 ...")
mistral_tokenizer = AutoTokenizer.from_pretrained(MISTRAL_MODEL_ID)
mistral_model = AutoModelForCausalLM.from_pretrained(
    MISTRAL_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
mistral_model.eval()
print("✅ Mistral loaded.")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## Cell 10 — Stage 2 + 3: Multi-turn Q&A + Simple Q&A with Mistral

Reads `paddy_disease_desc.jsonl`, generates both conversation types,  
and writes the final `paddy_disease.jsonl`.


In [ ]:
import json, torch
from pathlib import Path

WORKSHOP_DIR    = "/content/paddy-vlm-workshop"
TEMP_DESC_FILE  = Path(f"{WORKSHOP_DIR}/paddy_disease_desc.jsonl")
EXTERNAL_DIR    = Path(f"{WORKSHOP_DIR}/other_resources/External_Knowledge")
FINAL_OUT_FILE  = Path(f"{WORKSHOP_DIR}/paddy_disease.jsonl")
MAX_NEW_TOKENS_QA = 256   # 256 is enough for Q&A; was 512

def read_txt(path):
    p = Path(path)
    if not p.exists():
        return ""
    for enc in ("utf-8", "latin-1"):
        try:
            return p.read_text(encoding=enc).strip()
        except UnicodeDecodeError:
            pass
    with open(p, "rb") as f:
        return f.read().decode(errors="ignore").strip()

def call_mistral(system_prompt, user_content):
    messages = [{"role": "user", "content": f"{system_prompt}\n\n{user_content}"}]
    encoded  = mistral_tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(mistral_model.device)
    with torch.inference_mode():
        out = mistral_model.generate(
            encoded,
            max_new_tokens=MAX_NEW_TOKENS_QA,
            do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    generated = out[:, encoded.shape[1]:]
    return mistral_tokenizer.decode(generated[0], skip_special_tokens=True).strip()

# ── Resume: load already-completed image paths from FINAL_OUT_FILE ────────────
done_paths = set()
if FINAL_OUT_FILE.exists():
    with open(FINAL_OUT_FILE, "r", encoding="utf-8") as _f:
        for _line in _f:
            try:
                done_paths.add(json.loads(_line)["image_path"])
            except Exception:
                pass
    if done_paths:
        print(f"ℹ️  Resuming — {len(done_paths)} records already in output, skipping them.")

# ── Stage 2: Multi-turn Q&A ───────────────────────────────────────────────────
print("Stage 2 — Multi-turn Q&A ...")
multiturn_records = []
with open(TEMP_DESC_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line)
        if data.get("image_path") in done_paths:
            continue   # already processed in a previous run — skip
        ext  = read_txt(EXTERNAL_DIR / data["dataset"] / f"{data['class']}.txt")
        sys_msg  = (
            "You are an AI assistant specialized in agricultural topics. "
            "You are provided with a text description of a plant image, its attributes, "
            "and common agricultural knowledge. You do NOT have access to the actual image."
        )
        user_msg = (
            "Generate exactly 3 to 5 Q&A pairs.\n"
            "Each question starts with Q: and each answer with A:.\n"
            "No text outside the Q&A pairs.\n"
            "Focus on visual symptoms, disease, prevention.\n"
            "Format:\n  Q1: ...\n  A1: ...\n  Q2: ...\n  A2: ...\n\n"
            f"Image Description: {data['description']}\n"
            f"Attributes: {data['attributes']}\n"
            f"External Knowledge: {ext}"
        )
        try:
            mt = call_mistral(sys_msg, user_msg)
            data["external_knowledge"]     = ext
            data["multi_turn_conversation"] = mt
            multiturn_records.append(data)
        except Exception as e:
            print(f"  ERROR stage-2 {data['image_path']}: {e}")
        if len(multiturn_records) % 10 == 0 and multiturn_records:
            print(f"  [{len(multiturn_records)}] multi-turn done")

print(f"✅ Stage 2 complete — {len(multiturn_records)} new records.")


# ── Stage 3: Simple Q&A + save JSONL ─────────────────────────────────────────
print("\nStage 3 — Simple Q&A ...")
with open(FINAL_OUT_FILE, "a", encoding="utf-8") as out_f:   # "a" = append for resume
    for j, data in enumerate(multiturn_records):
        sys_msg  = (
            "You are an AI assistant specialized in agricultural topics. "
            "Generate 3-5 basic question-answer pairs about a plant image."
        )
        user_msg = (
            "Instructions:\n"
            "- Start each question with Q: and each answer with A:\n"
            "- Keep answers very short (name or label only, e.g. \"Blast\", \"Hispa\")\n"
            "- No full sentences or explanations in answers\n\n"
            f"Image Description: {data['description']}\n"
            f"Attributes: {data['attributes']}\n"
            f"External Knowledge: {data['external_knowledge']}\n"
            f"Class: {data['class']}, Dataset: {data['dataset']}"
        )
        try:
            sq = call_mistral(sys_msg, user_msg)
            data["simple_qa"] = sq
            out_f.write(json.dumps(data, ensure_ascii=False) + "\n")
            out_f.flush()   # persist after each record
        except Exception as e:
            print(f"  ERROR stage-3 {data['image_path']}: {e}")
        if (j + 1) % 10 == 0:
            print(f"  [{j+1}] simple-QA done")

total_final = len(done_paths) + len(multiturn_records)
print(f"\n✅ Dataset generation complete  →  {FINAL_OUT_FILE}  ({total_final} total records)")


## Cell 11 — Convert `paddy_disease.jsonl` to LLaVA Fine-tuning Format

Produces `paddy_disease_llava.json` in the official LLaVA conversation format.


In [ ]:
import json, uuid
from pathlib import Path

WORKSHOP_DIR = "/content/paddy-vlm-workshop"
INPUT_FILE   = Path(f"{WORKSHOP_DIR}/paddy_disease.jsonl")
OUTPUT_FILE  = Path(f"{WORKSHOP_DIR}/paddy_disease_llava.json")

def parse_qa_block(text):
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    pairs, q, a = [], None, None
    for line in lines:
        lower = line.lower()
        if lower[0] == "q" and ":" in line:
            if q and a:
                pairs.append((q, a))
                a = None
            q = line.split(":", 1)[1].strip()
        elif lower[0] == "a" and ":" in line:
            a = line.split(":", 1)[1].strip()
    if q and a:
        pairs.append((q, a))
    return pairs

output_records = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        rec   = json.loads(line)
        convs = []
        if rec.get("description"):
            convs.append({"from": "human", "value": "<image>\nDescribe the image."})
            convs.append({"from": "gpt",   "value": rec["description"].strip()})
        for q, a in parse_qa_block(rec.get("multi_turn_conversation", "")):
            convs.append({"from": "human", "value": q})
            convs.append({"from": "gpt",   "value": a})
        for q, a in parse_qa_block(rec.get("simple_qa", "")):
            convs.append({"from": "human", "value": q})
            convs.append({"from": "gpt",   "value": a})
        output_records.append({
            "id"           : str(uuid.uuid4()),
            "image"        : rec["image_path"],
            "conversations": convs,
        })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(output_records, f, ensure_ascii=False, indent=2)

print(f"✅ Converted {len(output_records)} records  →  {OUTPUT_FILE}")
if output_records:
    print(f"   Sample conversation turns: {len(output_records[0]['conversations'])}")


## Cell 12 — (Optional) Save Generated Files to Google Drive

Skips silently if Drive is not mounted.


In [ ]:
import shutil, os
from pathlib import Path

WORKSHOP_DIR = "/content/paddy-vlm-workshop"

try:
    drive_out = DRIVE_OUT  # set in Cell 6
except NameError:
    drive_out = "/content/PaddyVLM_outputs"

os.makedirs(drive_out, exist_ok=True)

for fname in ["paddy_disease_desc.jsonl", "paddy_disease.jsonl", "paddy_disease_llava.json"]:
    src = Path(WORKSHOP_DIR) / fname
    dst = os.path.join(drive_out, fname)
    if src.exists():
        shutil.copy(str(src), dst)
        size_kb = src.stat().st_size / 1024
        print(f"✅ Saved {fname} ({size_kb:.1f} KB)  →  {dst}")
    else:
        print(f"⚠️  {fname} not found — run Cells 8–11 first.")


## Cell 13 — Unload Mistral, Clone LLaVA Training Repo + Install Stack

Unloading Mistral reclaims ~5 GB VRAM needed for the training process.  
Then clones the official `haotian-liu/LLaVA` repository and installs its training dependencies.

> **flash-attn is skipped** — compiling from source takes 15–20 min on a free T4 and is not  
> required for LoRA training. Standard attention is used instead, with negligible quality impact  
> at this training scale. Total expected time: **~5 minutes**.


In [ ]:
import gc, torch, os

print("Unloading Mistral ...")
del mistral_model, mistral_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after unload: {torch.cuda.memory_allocated()/1e9:.2f} GB")

LLAVA_REPO = "/content/LLaVA"
if not os.path.exists(LLAVA_REPO):
    !git clone https://github.com/haotian-liu/LLaVA.git {LLAVA_REPO}
    print(f"✅ Cloned LLaVA repo to {LLAVA_REPO}")
else:
    print(f"LLaVA repo already at {LLAVA_REPO}")

%cd {LLAVA_REPO}
!ls


In [ ]:
%%capture
# flash-attn is intentionally skipped — compiling from source takes 15-20 min
# on a free Colab T4 and is not required for LoRA training.
!pip install -e . --quiet
!pip install -e ".[train]" --quiet
print("✅ LLaVA training stack installed (flash-attn skipped to save time).")


## Cell 14 — Prepare LLaVA Playground Data Directory

LLaVA's training script expects:
```
LLaVA/playground/data/paddy_disease_llava.json
LLaVA/playground/data/dataset/paddy_disease/<class>/*.jpg
```


In [ ]:
import os, shutil
from pathlib import Path

WORKSHOP_DIR   = "/content/paddy-vlm-workshop"
LLAVA_REPO     = "/content/LLaVA"
PLAYGROUND     = Path(f"{LLAVA_REPO}/playground/data")
PLAYGROUND_IMG = PLAYGROUND / "dataset"
os.makedirs(str(PLAYGROUND_IMG), exist_ok=True)

# Copy instruction JSON
src_json = Path(WORKSHOP_DIR) / "paddy_disease_llava.json"
dst_json = PLAYGROUND / "paddy_disease_llava.json"
if src_json.exists():
    shutil.copy(str(src_json), str(dst_json))
    print(f"✅ Copied {src_json.name}  →  {dst_json}")
else:
    raise FileNotFoundError(f"{src_json} not found — run Cell 11 first.")

# Symlink image folder to avoid copying GBs
src_img = Path(WORKSHOP_DIR) / "datasets" / "paddy_disease"
dst_img = PLAYGROUND_IMG / "paddy_disease"
if not dst_img.exists():
    if src_img.exists():
        os.symlink(str(src_img), str(dst_img))
        print(f"✅ Symlinked images: {src_img}  →  {dst_img}")
    else:
        print(f"⚠️  Dataset not found at {src_img}. Check Cell 5.")
else:
    print(f"Images already at {dst_img}")

print("\nPlayground:")
!ls -la {PLAYGROUND}


## Cell 15 — Generate LoRA Fine-tuning Script

Parameters tuned for **T4 (16 GB)** within the free Colab GPU limit.  
`NUM_EPOCHS = 1` — trains one full pass over the dataset (~10–20 min for 100 records).  
Raise to `3` for better quality if you have a paid Colab or extra GPU time.  
Adjust `LORA_R` or `BASE_MODEL` as needed.

> `--report_to none` disables WandB. Change to `wandb` if you want experiment tracking.


In [ ]:
import os

LLAVA_REPO      = "/content/LLaVA"
BASE_MODEL      = "liuhaotian/llava-v1.5-7b"
LORA_OUTPUT_DIR = "/content/drive/MyDrive/PaddyVLM_outputs/llava-paddy-lora"
NUM_EPOCHS      = 1    # 1 epoch fits free Colab T4 limit; raise to 3 for better quality
GRAD_ACCUM      = 4
LR              = "2e-4"
LORA_R          = 128
LORA_ALPHA      = 256

# Use local output if Drive is not mounted
try:
    if not os.path.exists("/content/drive/MyDrive"):
        LORA_OUTPUT_DIR = "/content/PaddyVLM_outputs/llava-paddy-lora"
except:
    LORA_OUTPUT_DIR = "/content/PaddyVLM_outputs/llava-paddy-lora"

os.makedirs(LORA_OUTPUT_DIR, exist_ok=True)

script = (
    "#!/bin/bash\n"
    "deepspeed llava/train/train_mem.py \\\n"
    f"    --lora_enable True \\\n"
    f"    --lora_r {LORA_R} \\\n"
    f"    --lora_alpha {LORA_ALPHA} \\\n"
    f"    --mm_projector_lr 2e-5 \\\n"
    f"    --deepspeed ./scripts/zero2.json \\\n"
    f"    --model_name_or_path {BASE_MODEL} \\\n"
    f"    --version v1 \\\n"
    f"    --data_path ./playground/data/paddy_disease_llava.json \\\n"
    f"    --image_folder ./playground/data/dataset \\\n"
    f"    --vision_tower openai/clip-vit-large-patch14-336 \\\n"
    f"    --mm_projector_type mlp2x_gelu \\\n"
    f"    --mm_vision_select_layer -2 \\\n"
    f"    --mm_use_im_start_end False \\\n"
    f"    --mm_use_im_patch_token False \\\n"
    f"    --image_aspect_ratio pad \\\n"
    f"    --group_by_modality_length True \\\n"
    f"    --bf16 True \\\n"
    f"    --output_dir {LORA_OUTPUT_DIR} \\\n"
    f"    --num_train_epochs {NUM_EPOCHS} \\\n"
    f"    --per_device_train_batch_size 1 \\\n"
    f"    --per_device_eval_batch_size 1 \\\n"
    f"    --gradient_accumulation_steps {GRAD_ACCUM} \\\n"
    f"    --evaluation_strategy no \\\n"
    f"    --save_strategy steps \\\n"
    f"    --save_steps 50000 \\\n"
    f"    --save_total_limit 1 \\\n"
    f"    --learning_rate {LR} \\\n"
    f"    --weight_decay 0. \\\n"
    f"    --warmup_ratio 0.03 \\\n"
    f"    --lr_scheduler_type cosine \\\n"
    f"    --logging_steps 1 \\\n"
    f"    --tf32 True \\\n"
    f"    --model_max_length 2048 \\\n"
    f"    --gradient_checkpointing True \\\n"
    f"    --dataloader_num_workers 4 \\\n"
    f"    --lazy_preprocess True \\\n"
    f"    --report_to none\n"
)

script_path = f"{LLAVA_REPO}/scripts/v1_5/finetune_paddy_lora.sh"
os.makedirs(os.path.dirname(script_path), exist_ok=True)
with open(script_path, "w") as fp:
    fp.write(script)
os.chmod(script_path, 0o755)
print(f"✅ Training script written to {script_path}")
print(f"   LoRA output dir: {LORA_OUTPUT_DIR}")
!cat {script_path}


## Cell 16 — Install DeepSpeed

In [ ]:
%%capture
!pip install -q deepspeed ninja
print("✅ DeepSpeed installed.")


## Cell 17 — Run LoRA Fine-tuning

With `MAX_IMAGES_PER_CLASS=10` (100 images) and `NUM_EPOCHS=1` on a free T4, expect **~15–25 min**.  
Checkpoints are saved to the output directory (Drive if mounted, local otherwise).

**Estimated total pipeline time on a free Colab T4 (fits within the ~4–5 h limit):**

| Step | Time |
|---|---|
| Cells 1–5 (setup, install, Kaggle download) | ~15 min |
| Cell 7 (load LLaVA 4-bit) | ~8 min |
| Cell 8 (Stage 1: 100 descriptions) | ~15 min |
| Cell 9 (swap to Mistral 4-bit) | ~8 min |
| Cell 10 (Stage 2+3: 100 Q&A records) | ~30 min |
| Cell 13 (clone + install LLaVA training stack) | ~5 min |
| **Cell 17 (LoRA fine-tune 1 epoch)** | ~20 min |
| **Total** | **~1 h 40 min** |


In [ ]:
import os
os.chdir("/content/LLaVA")
!bash scripts/v1_5/finetune_paddy_lora.sh


## Cell 18 — (Optional) Merge LoRA Weights into Base Model

In [ ]:
import os

LLAVA_REPO = "/content/LLaVA"

try:
    merged_base = DRIVE_OUT
except NameError:
    merged_base = "/content/PaddyVLM_outputs"

MERGED_MODEL_DIR = os.path.join(merged_base, "llava-paddy-merged")
os.makedirs(MERGED_MODEL_DIR, exist_ok=True)
os.chdir(LLAVA_REPO)

!python scripts/merge_lora_weights.py \
    --model-path "{LORA_OUTPUT_DIR}" \
    --model-base "{BASE_MODEL}" \
    --save-model-path "{MERGED_MODEL_DIR}"

print(f"✅ Merged model saved to {MERGED_MODEL_DIR}")


## Cell 19 — Quick Inference Test on a Sample Image

In [ ]:
import torch, glob, os
from transformers import LlavaProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from PIL import Image

WORKSHOP_DIR  = "/content/paddy-vlm-workshop"
HF_BASE_MODEL = "llava-hf/llava-1.5-7b-hf"

# ── Helpers ───────────────────────────────────────────────────────────────────
def is_valid_model_dir(path):
    """True only if path contains config.json (rules out empty makedirs folders)."""
    return os.path.isdir(path) and os.path.isfile(os.path.join(path, "config.json"))

def find_adapter_dir(root):
    """
    Return the directory that contains adapter_config.json, searching:
      1. root itself
      2. immediate subdirectories (e.g. checkpoint-500/)
      3. any deeper subdirectory (fallback)
    Returns None if not found.
    """
    if not os.path.isdir(root):
        return None
    # Check root first
    if os.path.isfile(os.path.join(root, "adapter_config.json")):
        return root
    # Check one level deep (most common: checkpoint-N/)
    for entry in sorted(os.listdir(root)):
        sub = os.path.join(root, entry)
        if os.path.isdir(sub) and os.path.isfile(os.path.join(sub, "adapter_config.json")):
            return sub
    # Deep search as last resort
    for dirpath, dirnames, filenames in os.walk(root):
        if "adapter_config.json" in filenames:
            return dirpath
    return None

try:
    merged_dir = MERGED_MODEL_DIR
except NameError:
    merged_dir = "/content/PaddyVLM_outputs/llava-paddy-merged"

try:
    lora_dir = LORA_OUTPUT_DIR
except NameError:
    lora_dir = "/content/PaddyVLM_outputs/llava-paddy-lora"

bnb_infer = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

if is_valid_model_dir(merged_dir):
    # ── Case 1: merged model available — load directly ────────────────────────
    print(f"Loading merged model from {merged_dir} ...")
    infer_proc  = LlavaProcessor.from_pretrained(merged_dir)
    infer_model = LlavaForConditionalGeneration.from_pretrained(
        merged_dir,
        quantization_config=bnb_infer,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    print("✅ Merged model loaded.")

else:
    adapter_dir = find_adapter_dir(lora_dir)
    if adapter_dir:
        # ── Case 2: LoRA adapters found — load HF base + PEFT ────────────────
        from peft import PeftModel
        print(f"Loading HF base + LoRA adapters ...")
        print(f"  Base   : {HF_BASE_MODEL}")
        print(f"  Adapter: {adapter_dir}")

        infer_proc = LlavaProcessor.from_pretrained(HF_BASE_MODEL)
        base_model = LlavaForConditionalGeneration.from_pretrained(
            HF_BASE_MODEL,
            quantization_config=bnb_infer,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )
        infer_model = PeftModel.from_pretrained(base_model, adapter_dir)
        print("✅ Base model + LoRA adapters loaded.")

    else:
        # ── Case 3: nothing trained yet — load unmodified base model ─────────
        print(f"⚠️  No fine-tuned model found (adapter_config.json not found under {lora_dir}).")
        print(f"   Loading unmodified base: {HF_BASE_MODEL}")
        print("   Run Cells 15–17 (and optionally 18) to produce a fine-tuned checkpoint.")
        infer_proc  = LlavaProcessor.from_pretrained(HF_BASE_MODEL)
        infer_model = LlavaForConditionalGeneration.from_pretrained(
            HF_BASE_MODEL,
            quantization_config=bnb_infer,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )
        print("✅ Base model loaded (not fine-tuned).")

infer_model.eval()
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Run inference on a sample image ───────────────────────────────────────────
img_files = (
    glob.glob(f"{WORKSHOP_DIR}/datasets/paddy_disease/**/*.jpg",  recursive=True) or
    glob.glob(f"{WORKSHOP_DIR}/datasets/paddy_disease/**/*.jpeg", recursive=True)
)

if img_files:
    image = Image.open(img_files[0]).convert("RGB")
    print(f"\nTest image: {img_files[0]}")

    prompt = "USER: <image>\nWhat disease does this paddy plant have? Describe the symptoms and suggest a treatment. ASSISTANT:"
    inputs = infer_proc(text=prompt, images=image, return_tensors="pt").to(infer_model.device)
    with torch.inference_mode():
        out = infer_model.generate(**inputs, max_new_tokens=400, do_sample=False)
    answer = infer_proc.decode(out[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()

    print("\n" + "=" * 60)
    print("PaddyVLM response:")
    print("=" * 60)
    print(answer)
else:
    print("No test images found — check that Cell 5 ran successfully.")


## Cell 20 — Output Summary

In [ ]:
import os
from pathlib import Path

WORKSHOP_DIR = "/content/paddy-vlm-workshop"

try:
    lora_dir   = Path(LORA_OUTPUT_DIR)
    merged_dir = Path(MERGED_MODEL_DIR)
except NameError:
    lora_dir   = Path("/content/PaddyVLM_outputs/llava-paddy-lora")
    merged_dir = Path("/content/PaddyVLM_outputs/llava-paddy-merged")

files_to_check = {
    "Image descriptions (JSONL)": Path(WORKSHOP_DIR) / "paddy_disease_desc.jsonl",
    "Full dataset (JSONL)"      : Path(WORKSHOP_DIR) / "paddy_disease.jsonl",
    "LLaVA format (JSON)"       : Path(WORKSHOP_DIR) / "paddy_disease_llava.json",
    "LoRA checkpoint"           : lora_dir,
    "Merged model"              : merged_dir,
}

print("PaddyVLM Pipeline — Output Summary")
print("=" * 58)
for label, path in files_to_check.items():
    exists = path.exists()
    size   = ""
    if exists and path.is_file():
        size = f"({path.stat().st_size / 1024:.1f} KB)"
    elif exists and path.is_dir():
        n = sum(1 for _ in path.rglob("*") if Path(_).is_file())
        size = f"({n} files)"
    status = "OK" if exists else "NOT FOUND"
    print(f"  [{status:9s}] {label:35s} {size}")
    if exists:
        print(f"               → {path}")
print("=" * 58)
